In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

import folium
from geopy.geocoders import Nominatim
import time
import matplotlib
import matplotlib.cm as cm
from pathlib import Path

from matplotlib import font_manager, rc
import os

# Windows 기본 한글 폰트 경로
font_path = "C:/Windows/Fonts/malgun.ttf"

font = font_manager.FontProperties(fname=font_path).get_name()
rc("font", family=font)

# 마이너스 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

# 데이터(all_data_v1_update) 로드

In [2]:
# 현재 노트북 위치
BASE_DIR = Path().resolve()

# 프로젝트 루트 (notebooks의 상위 폴더)
PROJECT_ROOT = BASE_DIR.parent

# data 폴더
DATA_DIR = PROJECT_ROOT / "data"

# data 파일
path1 = DATA_DIR / "all_data_v1_update.csv"

# data load
df = pd.read_csv(path1, encoding="utf-8")

# 성별 컬럼 생성

In [3]:
# -----------------------
# 1) 성별 컬럼 정의
# -----------------------
female_cols = [c for c in df.columns if c.startswith("여성")]
male_cols   = [c for c in df.columns if c.startswith("남성")]

# -----------------------
# 2) 행 단위 성별 합계
# -----------------------
df["여성_합계"] = df[female_cols].sum(axis=1)
df["남성_합계"] = df[male_cols].sum(axis=1)


# 연령별 컬럼 생성

In [4]:
# -----------------------
# 1) 연령대 컬럼 정의
# -----------------------
age_cols = {
    "20대": [c for c in df.columns if "20대" in c],
    "30대": [c for c in df.columns if "30대" in c],
    "40대": [c for c in df.columns if "40대" in c],
    "50대": [c for c in df.columns if "50대" in c],
    "60대": [c for c in df.columns if "60대" in c],

}

# -----------------------
# 2) 행(row) 단위 연령대 합계
# -----------------------
for age, cols in age_cols.items():
    df[f"{age}_합계"] = df[cols].sum(axis=1)

# 배달여부 컬럼 생성

In [5]:
df["배달여부"] = (
    df.groupby("가맹점구분번호")["배달매출금액 비율"]
      .transform(lambda x: (x != 0).any())
      .astype(int)
)

In [6]:
df

,가맹점구분번호,가맹점주소,업종,상권,개설일,폐업일,폐업여부,기준년월,가맹점 운영개월수 구간,매출금액 구간,...,직장 이용 고객 비율,유동인구 이용 고객 비율,여성_합계,남성_합계,20대_합계,30대_합계,40대_합계,50대_합계,60대_합계,배달여부
0,16184E93D9,서울특별시 성동구 마장동,축산물,마장동,2013-03-20,NaN,False,2024-05-01,2,3,...,7.1,42.9,32.0000,68.0000,6.1,13.3,16.3,28.6,35.7,0
1,16184E93D9,서울특별시 성동구 마장동,축산물,마장동,2013-03-20,NaN,False,2023-04-01,2,4,...,6.3,68.8,39.6000,60.4000,6.6,18.5,15.6,28.0,31.3,0
2,16184E93D9,서울특별시 성동구 마장동,축산물,마장동,2013-03-20,NaN,False,2023-08-01,2,3,...,0.0,82.4,37.0000,63.0000,6.2,17.8,16.9,27.7,31.4,0
3,16184E93D9,서울특별시 성동구 마장동,축산물,마장동,2013-03-20,NaN,False,2024-02-01,2,3,...,5.3,78.9,30.8000,69.2000,6.8,15.0,15.0,31.6,31.6,0
4,16184E93D9,서울특별시 성동구 마장동,축산물,마장동,2013-03-20,NaN,False,2024-06-01,2,3,...,0.0,73.3,35.4645,64.4355,6.1,12.2,16.8,28.1,36.7,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85964,58B8C943BF,서울특별시 성동구 매봉길 50,아이스크림/빙수,(상권 외 지역),2023-03-03,NaN,False,2023-03-01,6,4,...,4.7,22.5,55.2000,44.8000,17.1,34.1,27.1,15.5,6.2,0
85965,58B8C943BF,서울특별시 성동구 매봉길 50,아이스크림/빙수,(상권 외 지역),2023-03-03,NaN,False,2024-04-01,5,4,...,2.0,30.7,51.9000,48.1000,20.2,33.1,28.5,11.2,7.0,0
85966,58B8C943BF,서울특별시 성동구 매봉길 50,아이스크림/빙수,(상권 외 지역),2023-03-03,NaN,False,2024-10-01,5,4,...,1.6,26.8,47.9479,52.1521,19.8,32.3,29.6,12.0,6.4,0
85967,58B8C943BF,서울특별시 성동구 매봉길 50,아이스크림/빙수,(상권 외 지역),2023-03-03,NaN,False,2024-06-01,5,3,...,2.9,26.7,50.7000,49.3000,20.4,33.0,29.0,11.2,6.4,0


# 분기별로 데이터 압축

In [7]:
# 분기 컬럼 생성
df["기준년월"] = pd.to_datetime(df["기준년월"])
df["연도"] = df["기준년월"].dt.year
df["분기"] = df["기준년월"].dt.quarter

In [8]:
# 지정한 컬럼
# target_cols = [
#     "가맹점 운영개월수 구간",
#     "매출금액 구간",
#     "유니크 고객 수 구간",
#     "취소율 구간",
#     "배달매출금액 비율",
#     "배달여부",
#     "재방문 고객 비중",
#     '폐업여부',
# ]

target_cols = ['가맹점 운영개월수 구간', '매출금액 구간', '매출건수 구간', '유니크 고객 수 구간', '객단가 구간', '취소율 구간',
       '배달매출금액 비율', '동일 업종 매출금액 비율', '동일 업종 매출건수 비율', '동일 업종 내 매출 순위 비율',
       '동일 상권 내 매출 순위 비율', '동일 업종 내 해지 가맹점 비중', '동일 상권 내 해지 가맹점 비중',
       '여성_합계', '남성_합계', '20대_합계', '30대_합계', '40대_합계', '50대_합계','60대_합계',
       '재방문 고객 비중', '신규 고객 비중',
       '거주 이용 고객 비율', '직장 이용 고객 비율', '유동인구 이용 고객 비율','폐업여부','배달여부']


In [9]:
# 상점×분기 압축 (평균/표준편차)
summary_df = (
    df.groupby(["가맹점구분번호", "연도", "분기"], dropna=False)[target_cols]
      .agg(["mean", "std"])
)

In [10]:
# 컬럼명 깔끔하게: (컬럼, mean/std) → "컬럼_mean" 형태
summary_df.columns = [f"{col}_{stat}" for col, stat in summary_df.columns]
summary_df = summary_df.reset_index()

# 선택: 표준편차가 NaN(분기 데이터 1개뿐이면 std가 NaN)인 경우 0으로
std_cols = [c for c in summary_df.columns if c.endswith("_std")]
summary_df[std_cols] = summary_df[std_cols].fillna(0)

summary_df.head()

,가맹점구분번호,연도,분기,가맹점 운영개월수 구간_mean,가맹점 운영개월수 구간_std,매출금액 구간_mean,매출금액 구간_std,매출건수 구간_mean,매출건수 구간_std,유니크 고객 수 구간_mean,...,거주 이용 고객 비율_mean,거주 이용 고객 비율_std,직장 이용 고객 비율_mean,직장 이용 고객 비율_std,유동인구 이용 고객 비율_mean,유동인구 이용 고객 비율_std,폐업여부_mean,폐업여부_std,배달여부_mean,배달여부_std
0,000F03E44A,2023,1,5.000000,0.00000,6.000000,0.00000,5.666667,0.57735,5.666667,...,0.0,0.0,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0
1,000F03E44A,2023,2,5.000000,0.00000,6.000000,0.00000,5.000000,0.00000,5.000000,...,0.0,0.0,33.333333,57.735027,66.666667,57.735027,0.0,0.0,1.0,0.0
2,000F03E44A,2023,3,5.000000,0.00000,6.000000,0.00000,5.666667,0.57735,5.666667,...,0.0,0.0,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0
3,000F03E44A,2023,4,4.333333,0.57735,6.000000,0.00000,5.666667,0.57735,5.666667,...,0.0,0.0,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0
4,000F03E44A,2024,1,4.000000,0.00000,5.666667,0.57735,5.333333,0.57735,5.333333,...,0.0,0.0,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0


# 상권 활성화지수 컬럼 생성

## 상권_유동인구_수 

In [11]:
from sklearn.preprocessing import StandardScaler

trade_area_path = DATA_DIR / "서울시 상권분석서비스(길단위인구-상권).csv"
trade_area_info = pd.read_csv(trade_area_path, encoding="cp949")

In [12]:
# 0) 기준 분기 & 성동구 상권 목록
target_quarters = [20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244]
seongdong_area = df["상권"].unique()

# 1) 외부 유동인구 데이터 전처리 (필터링 + 연도/분기 분리 + 컬럼 정리)
trade_area_population = (
    trade_area_info.loc[
        trade_area_info["기준_년분기_코드"].isin(target_quarters)
        & trade_area_info["상권_코드_명"].isin(seongdong_area),
        ["상권_코드_명", "기준_년분기_코드", "총_유동인구_수"]
    ]
    .assign(
        연도=lambda x: x["기준_년분기_코드"].astype(str).str[:4].astype(int),
        분기=lambda x: x["기준_년분기_코드"].astype(str).str[4:].astype(int),
        상권=lambda x: x["상권_코드_명"]
    )
    .drop(columns=["상권_코드_명", "기준_년분기_코드"])
    .rename(columns={"총_유동인구_수": "상권_유동인구_수"})
)

## 상권_소비력

In [13]:
# 1) 상권별 평균값 집계
area_agg = (
    df.groupby("상권", as_index=False)
        .agg({
            "매출금액 구간": "mean",
            "유니크 고객 수 구간": "mean",
        })
)

# # 2) 순위/구간 점수화
#    - 매출금액, 고객 수 구간: 1~6 → 0~100 선형 변환
area_agg["매출_점수"] = (6 - area_agg["매출금액 구간"]) / 5 * 100
area_agg["고객수_점수"] = (6 - area_agg["유니크 고객 수 구간"]) / 5 * 100


# 3) 상권 소비력 계산
#    - 매출, 고객 수, 업종 내 순위 점수 평균

sel_col = ["매출_점수", "고객수_점수"] 
area_agg["상권_소비력"] = area_agg[sel_col].mean(axis=1)
area_active= trade_area_population.merge(area_agg[["상권","상권_소비력"]])

## 상권_활성화지수

In [14]:
# 1) 표준화
scaler = StandardScaler()
scaled = scaler.fit_transform(area_active[["상권_유동인구_수", "상권_소비력"]])

# 2) 음수값 처리 및 기하평균 계산
scaled_shifted = scaled - scaled.min(axis=0) + 1e-9
area_active["상권_활성화지수"] = np.sqrt(scaled_shifted[:, 0] * scaled_shifted[:, 1])
area_active = area_active.drop(columns=["상권_유동인구_수", "상권_소비력"])

In [15]:
# 2) 병합 (상권/연도/분기 기준)
df_first = (
    df[["가맹점구분번호",'상권']]
    .drop_duplicates(subset="가맹점구분번호", keep="first")
)

final_df = summary_df.merge(
    df_first,
    on="가맹점구분번호",
    how="left"
)

merged_df = final_df.merge(
    area_active,
    on=["상권", "연도", "분기"],
    how="left"
)
# 결측치 처리
merged_df["상권_활성화지수"] = merged_df["상권_활성화지수"].fillna(
    merged_df["상권_활성화지수"].mean()
)
merged_df = merged_df.drop(columns=["상권"])

# 
summary_df = merged_df.copy()


In [16]:
summary_df

,가맹점구분번호,연도,분기,가맹점 운영개월수 구간_mean,가맹점 운영개월수 구간_std,매출금액 구간_mean,매출금액 구간_std,매출건수 구간_mean,매출건수 구간_std,유니크 고객 수 구간_mean,...,거주 이용 고객 비율_std,직장 이용 고객 비율_mean,직장 이용 고객 비율_std,유동인구 이용 고객 비율_mean,유동인구 이용 고객 비율_std,폐업여부_mean,폐업여부_std,배달여부_mean,배달여부_std,상권_활성화지수
0,000F03E44A,2023,1,5.000000,0.00000,6.000000,0.00000,5.666667,0.57735,5.666667,...,0.000000,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0,1.409280
1,000F03E44A,2023,2,5.000000,0.00000,6.000000,0.00000,5.000000,0.00000,5.000000,...,0.000000,33.333333,57.735027,66.666667,57.735027,0.0,0.0,1.0,0.0,1.399716
2,000F03E44A,2023,3,5.000000,0.00000,6.000000,0.00000,5.666667,0.57735,5.666667,...,0.000000,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0,1.307137
3,000F03E44A,2023,4,4.333333,0.57735,6.000000,0.00000,5.666667,0.57735,5.666667,...,0.000000,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0,1.304933
4,000F03E44A,2024,1,4.000000,0.00000,5.666667,0.57735,5.333333,0.57735,5.333333,...,0.000000,0.000000,0.000000,100.000000,0.000000,0.0,0.0,1.0,0.0,1.264817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29020,FFF0DAC445,2023,4,3.000000,0.00000,1.666667,0.57735,2.666667,0.57735,2.000000,...,2.095233,15.366667,6.585084,69.133333,8.631531,0.0,0.0,0.0,0.0,3.272184
29021,FFF0DAC445,2024,1,3.000000,0.00000,1.333333,0.57735,3.000000,0.00000,2.000000,...,2.443358,16.366667,1.137248,66.466667,2.557994,0.0,0.0,0.0,0.0,3.276181
29022,FFF0DAC445,2024,2,3.000000,0.00000,2.000000,0.00000,3.000000,0.00000,2.666667,...,1.178983,17.233333,1.357694,65.000000,2.291288,0.0,0.0,0.0,0.0,3.266043
29023,FFF0DAC445,2024,3,3.000000,0.00000,1.333333,0.57735,2.666667,0.57735,2.333333,...,4.135618,18.000000,2.457641,61.466667,2.267892,0.0,0.0,0.0,0.0,3.230927


# 폐업 여부에 유의미하게 영향을 준 변수

In [24]:
summary_df = summary_df.rename(columns={"배달여부_mean": "배달여부",})

In [25]:
# 폐업 여부 기준 분리
df_open = summary_df[summary_df["폐업여부_mean"] == 0]
df_closed = summary_df[summary_df["폐업여부_mean"] == 1]

In [26]:
# test_vars = [
#     "가맹점 운영개월수 구간_mean",
#     "매출금액 구간_mean",
#     "유니크 고객 수 구간_mean",
#     "취소율 구간_mean",
#     "배달매출금액 비율_mean",
#     "배달여부_mean",
#     "재방문 고객 비중_mean",
#     "가맹점 운영개월수 구간_std",
#     "매출금액 구간_std",
#     "유니크 고객 수 구간_std",
#     "취소율 구간_std",
#     "배달매출금액 비율_std",
#     "재방문 고객 비중_std",
# ]

exclude_cols = ['가맹점구분번호', '연도', '분기','폐업여부_mean', '폐업여부_std']

test_vars = summary_df.columns.difference(exclude_cols)


In [27]:
summary_df.columns

Index(['가맹점구분번호', '연도', '분기', '가맹점 운영개월수 구간_mean', '가맹점 운영개월수 구간_std',
       '매출금액 구간_mean', '매출금액 구간_std', '매출건수 구간_mean', '매출건수 구간_std',
       '유니크 고객 수 구간_mean', '유니크 고객 수 구간_std', '객단가 구간_mean', '객단가 구간_std',
       '취소율 구간_mean', '취소율 구간_std', '배달매출금액 비율_mean', '배달매출금액 비율_std',
       '동일 업종 매출금액 비율_mean', '동일 업종 매출금액 비율_std', '동일 업종 매출건수 비율_mean',
       '동일 업종 매출건수 비율_std', '동일 업종 내 매출 순위 비율_mean', '동일 업종 내 매출 순위 비율_std',
       '동일 상권 내 매출 순위 비율_mean', '동일 상권 내 매출 순위 비율_std',
       '동일 업종 내 해지 가맹점 비중_mean', '동일 업종 내 해지 가맹점 비중_std',
       '동일 상권 내 해지 가맹점 비중_mean', '동일 상권 내 해지 가맹점 비중_std', '여성_합계_mean',
       '여성_합계_std', '남성_합계_mean', '남성_합계_std', '20대_합계_mean', '20대_합계_std',
       '30대_합계_mean', '30대_합계_std', '40대_합계_mean', '40대_합계_std', '50대_합계_mean',
       '50대_합계_std', '60대_합계_mean', '60대_합계_std', '재방문 고객 비중_mean',
       '재방문 고객 비중_std', '신규 고객 비중_mean', '신규 고객 비중_std', '거주 이용 고객 비율_mean',
       '거주 이용 고객 비율_std', '직장 이용 고객 비율_mean', '직장 이용 고객 비율_std',
       '유동인구 

## Mann–Whitney U test

In [29]:
# Mann–Whitney U test
from scipy.stats import mannwhitneyu

results = []

for var in test_vars:
    stat, p = mannwhitneyu(
        df_open[var],
        df_closed[var],
        alternative="two-sided"
    )
    
    results.append({
        "변수": var,
        "U_statistic": stat,
        "p_value": p
    })

test_result_df = pd.DataFrame(results).sort_values("p_value")
test_result_df


# from scipy.stats import mannwhitneyu
# import numpy as np
# import pandas as pd

# results = []

# for var in test_vars:
#     # Mann–Whitney U test
#     stat, p = mannwhitneyu(
#         df_open[var],
#         df_closed[var],
#         alternative="two-sided"
#     )
    
#     # 중앙값 계산
#     median_open = df_open[var].median()
#     median_closed = df_closed[var].median()
    
#     # 방향성 판단
#     if median_open > median_closed:
#         direction = "비폐업 > 폐업"
#     elif median_open < median_closed:
#         direction = "폐업 > 비폐업"
#     else:
#         direction = "차이 없음"
    
#     results.append({
#         "변수": var,
#         "U_statistic": stat,
#         "p_value": p,
#         "median_비폐업": median_open,
#         "median_폐업": median_closed,
#         "비교 방향 (median)": direction
#     })

# test_result_df = (
#     pd.DataFrame(results)
#       .sort_values("p_value")
#       .reset_index(drop=True)
# )

# test_result_df


,변수,U_statistic,p_value
16,남성_합계_mean,10105010.5,2.436891e-15
41,여성_합계_mean,6999860.0,1.398842e-13
10,가맹점 운영개월수 구간_mean,7025732.5,1.458074e-13
35,배달매출금액 비율_std,7372007.5,1.315902e-11
8,60대_합계_mean,9760765.5,4.989146e-10
0,20대_합계_mean,7252628.0,7.659789e-10
34,배달매출금액 비율_mean,7479557.0,3.105232e-09
36,배달여부,7707459.5,2.805575e-06
2,30대_합계_mean,7610576.0,1.159941e-05
6,50대_합계_mean,9381003.0,1.383620e-05


In [30]:
# p-value < 0.05인 변수 추출
significant_df = test_result_df[test_result_df["p_value"] < 0.05]
significant_df['변수'].tolist()

['남성_합계_mean',
 '여성_합계_mean',
 '가맹점 운영개월수 구간_mean',
 '배달매출금액 비율_std',
 '60대_합계_mean',
 '20대_합계_mean',
 '배달매출금액 비율_mean',
 '배달여부',
 '30대_합계_mean',
 '50대_합계_mean',
 '객단가 구간_mean',
 '상권_활성화지수',
 '60대_합계_std',
 '동일 업종 내 해지 가맹점 비중_std',
 '객단가 구간_std']

## Wilcoxon rank-sum test

In [22]:
# Wilcoxon rank-sum test
from scipy.stats import ranksums
import pandas as pd

results = []

for var in test_vars:
    stat, p = ranksums(
        df_open[var],
        df_closed[var]
    )
    
    results.append({
        "변수": var,
        "Z_statistic": stat,   # ranksums는 Z 통계량
        "p_value": p
    })

test_result_df = pd.DataFrame(results).sort_values("p_value")
test_result_df


,변수,Z_statistic,p_value
16,남성_합계_mean,7.916756,2.437871e-15
41,여성_합계_mean,-7.396360,1.399683e-13
10,가맹점 운영개월수 구간_mean,-7.268769,3.627779e-13
8,60대_합계_mean,6.219105,4.999998e-10
0,20대_합계_mean,-6.149829,7.756649e-10
35,배달매출금액 비율_std,-5.561107,2.680695e-08
34,배달매출금액 비율_mean,-5.030724,4.886315e-07
2,30대_합계_mean,-4.384601,1.161986e-05
6,50대_합계_mean,4.346298,1.384546e-05
36,배달여부_mean,-3.906818,9.351961e-05


## Welch t-test

In [31]:
# Welch t-test
from scipy.stats import ttest_ind
import pandas as pd

results = []

for var in test_vars:
    stat, p = ttest_ind(
        df_open[var],
        df_closed[var],
        equal_var=False,      # ⭐ Welch's t-test 핵심
        nan_policy="omit"     # 결측치 안전 처리
    )
    
    results.append({
        "변수": var,
        "t_statistic": stat,
        "p_value": p
    })

test_result_df = pd.DataFrame(results).sort_values("p_value")
test_result_df


,변수,t_statistic,p_value
10,가맹점 운영개월수 구간_mean,-9.368325,1.262503e-19
8,60대_합계_mean,7.557587,1.430469e-13
16,남성_합계_mean,6.733723,3.757082e-11
41,여성_합계_mean,-5.954561,4.353747e-09
0,20대_합계_mean,-5.697110,1.883457e-08
6,50대_합계_mean,5.272393,1.845828e-07
35,배달매출금액 비율_std,-5.085437,4.878683e-07
36,배달여부,-4.527877,7.144658e-06
34,배달매출금액 비율_mean,-4.402928,1.258829e-05
2,30대_합계_mean,-4.164056,3.562595e-05
